# Consolidate Collection Lineage Evidence JSON Files

This notebook combines the per-collection lineage evidence files produced by
**Collection_Lineage_via_ATBD_Skill** into one evidence file per collection folder. It's the JSON
counterpart of `consolidate_lineage.ipynb`, which does the same for the lineage CSVs.

**Input**
- `Reports/<DAAC>/<FOLDER>/<COLLECTION>_lineage_evidence.json`: one file per collection

Every `*_suite_lineage_evidence.json` roll-up is **ignored**. Those files aren't read and aren't included in the output.

**Output**
- `Reports/<DAAC>/<FOLDER>_lineage_evidence.json`, written in the DAAC directory next to its source folder.
  For example, `Reports/ASF/OPERA/*_lineage_evidence.json` becomes `Reports/ASF/OPERA_lineage_evidence.json`.

**Output structure**

| Key | Content |
|---|---|
| `consolidated` | Provenance: data center, folder, generation time, this notebook, source file list, counts |
| `collections` | One entry per per-collection evidence file, in file-name order. Each entry is the original JSON **unchanged**, with a `source_file` key added at the front |

In [1]:
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# ---- Inputs ---------------------------------------------------------------
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
REPORTS_DIR = REPO_ROOT / "Reports"
DAACS = ["ASF", "GES_DISC", "GHRC_DAAC", "LANCEMODIS", "LARC_CLOUD", "LPCLOUD", "LPDAAC_ECS"]
NOTEBOOK = "Notebooks/consolidate_lineage_evidence.ipynb"

# ---- Options --------------------------------------------------------------
DRY_RUN = False   # True = build and summarize without writing any JSON files

## Step 1: Discover the collection folders

Each subfolder of `Reports/<DAAC>/` is one collection suite. Only per-collection `*_lineage_evidence.json`
files are combined. Files that match `*_suite_lineage_evidence.json` are skipped. They're counted below only
to show what was ignored.

In [2]:
folders = []
for daac in DAACS:
    for folder in sorted(p for p in (REPORTS_DIR / daac).iterdir() if p.is_dir()):
        files = sorted(f for f in folder.glob("*_lineage_evidence.json")
                       if not f.name.endswith("_suite_lineage_evidence.json"))
        if files:
            folders.append((daac, folder, files))

ignored = sorted(REPORTS_DIR.glob("*/*/*_suite_lineage_evidence.json"))
print(f"{'DAAC':<11} {'Folder':<15} {'files':>5}")
for daac, folder, files in folders:
    print(f"{daac:<11} {folder.name:<15} {len(files):>5}")
print(f"\n{len(folders)} folders, {sum(len(f[2]) for f in folders)} per-collection files")
print(f"{len(ignored)} *_suite_lineage_evidence.json files ignored")

DAAC        Folder          files
ASF         ARIA                1
ASF         OPERA               8
ASF         SENTINEL-1          2
ASF         SENTINEL-1C        14
ASF         SENTINEL-1D        14
ASF         UAVSAR              2
GES_DISC    AIRS               39
GES_DISC    CAR                16
GES_DISC    LHSAS               3
GES_DISC    MEaSUREs_OMI        4
GES_DISC    MINDS               1
GES_DISC    SNDR               81
GES_DISC    TOMS                2
GES_DISC    TOVS                9
GES_DISC    TROPOMI             2
GHRC_DAAC   glmmth              1
LANCEMODIS  MOD09               6
LANCEMODIS  VIIRS-NOAA20        6
LANCEMODIS  VIIRS-NOAA21        2
LANCEMODIS  VIIRS-SNPP          6
LARC_CLOUD  AIRMISR             2
LARC_CLOUD  ARESE               1
LARC_CLOUD  ATOST               1
LARC_CLOUD  AirMSPI             6
LARC_CLOUD  CALIPSO            35
LARC_CLOUD  CATS               16
LARC_CLOUD  CERES             104
LARC_CLOUD  CLAMS               3
LARC_CLOUD  DS

## Step 2: Validate the inputs

Before merging, check that:
1. every file parses as a JSON object;
2. every per-collection file has a `target` with `short_name` and `version`;
3. no collection (`short_name` + `version`) appears twice in a folder.

In [3]:
def load(path):
    data = json.loads(path.read_text())
    if not isinstance(data, dict):
        raise ValueError(f"{path}: top level is {type(data).__name__}, expected an object")
    return data

problems = []
loaded = {}   # folder path -> list of (file, data)

for daac, folder, files in folders:
    docs = [(f, load(f)) for f in files]
    keys = Counter()
    for f, d in docs:
        t = d.get("target") or {}
        if not t.get("short_name") or not t.get("version"):
            problems.append(f"{f.relative_to(REPORTS_DIR)}: target is missing short_name/version")
        keys[(t.get("short_name"), t.get("version"))] += 1
    problems += [f"{folder.relative_to(REPORTS_DIR)}: {k} appears {n} times" for k, n in keys.items() if n > 1]
    loaded[folder] = docs

print(f"Parsed {sum(len(v) for v in loaded.values())} per-collection files")
print(f"{len(problems)} problems found")
for p in problems:
    print("  ", p)

Parsed 747 per-collection files
402 problems found
   GES_DISC/SNDR/SNDR13CHRP1AQCal_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDR13CHRP1J1Cal_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDR13CHRP1SNCal_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDR13CHRP1_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL2CCPCCR_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL2CCPRET_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL2CPS_V2.1_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL2JSFRET_V2_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL2PLEVCPS_V2.1_lineage_evidence.json: target is missing short_name/version
   GES_DISC/SNDR/SNDRAQIL3CDCCP_V2_lineage_evidence.json: target is missing short_na

### 2b. Per-collection schema variants

The per-collection files were written by several versions of the skill, so they don't all have the same
top-level sections. They are merged as-is. This table shows which optional sections occur and how often.

In [4]:
all_docs = [d for docs in loaded.values() for _, d in docs]
key_counts = Counter(k for d in all_docs for k in d)
print(f"{'Top-level key':<32} {'files':>5}")
for k, n in sorted(key_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"{k:<32} {n:>5}{'' if n == len(all_docs) else '   (optional)'}")

Top-level key                    files
excluded                           747
source_documents                   747
target                             747
atbd_search                        388   (optional)
csv                                388   (optional)
generated                          388   (optional)
granule_level_check                388   (optional)
lineage_graph                      388   (optional)
metadata_findings                  388   (optional)
parameters                         388   (optional)
relationships                      388   (optional)
skill                              388   (optional)
atbd_search_depth                  359   (optional)
csv_validation                     359   (optional)
generated_at                       359   (optional)
rows                               359   (optional)
existing_associations_check        347   (optional)
existing_metadata_associations     339   (optional)
include_low_confidence             339   (optional)
skills      

## Step 3: Consolidate and write `<FOLDER>_lineage_evidence.json`

Collections are in file-name order. The original content is not changed. Note that some files still have
paths from older layouts in their `csv.path` field (e.g. `Reports/OPERA_L2_CSLC-S1_V1_V1_lineage.csv`).
Those are left as they are, and `source_file` gives the file's real location.

In [5]:
generated = datetime.now(timezone.utc).isoformat(timespec="seconds")
results = []

for daac, folder, files in folders:
    docs = loaded[folder]
    collections = [{"source_file": str(f.relative_to(REPO_ROOT)), **d} for f, d in docs]
    n_rel = sum(len(d.get("relationships") or []) for _, d in docs)
    out = {
        "consolidated": {
            "generated": generated,
            "notebook": NOTEBOOK,
            "data_center": daac,
            "group": folder.name,
            "collection_count": len(collections),
            "relationship_count": n_rel,
            "source_files": [c["source_file"] for c in collections],
        },
        "collections": collections,
    }
    out_path = folder.parent / f"{folder.name}_lineage_evidence.json"
    if not DRY_RUN:
        out_path.write_text(json.dumps(out, indent=2, ensure_ascii=False) + "\n")
    results.append({"daac": daac, "folder": folder.name, "out": out_path, "files": len(collections),
                    "relationships": n_rel, "docs": docs})

action = "Would write" if DRY_RUN else "Wrote"
print(f"{'Output file':<52} {'colls':>5} {'rels':>5} {'size':>8}")
for r in results:
    size = f"{r['out'].stat().st_size / 1024:,.0f} KB" if r["out"].exists() else "-"
    print(f"{str(r['out'].relative_to(REPO_ROOT)):<52} {r['files']:>5} {r['relationships']:>5} "
          f"{size:>8}")
print(f"\n{action} {len(results)} files")

Output file                                          colls  rels     size
Reports/ASF/ARIA_lineage_evidence.json                   1     6    38 KB
Reports/ASF/OPERA_lineage_evidence.json                  8    40   210 KB
Reports/ASF/SENTINEL-1_lineage_evidence.json             2     8    49 KB
Reports/ASF/SENTINEL-1C_lineage_evidence.json           14   100   371 KB
Reports/ASF/SENTINEL-1D_lineage_evidence.json           14    98   384 KB
Reports/ASF/UAVSAR_lineage_evidence.json                 2    15   359 KB
Reports/GES_DISC/AIRS_lineage_evidence.json             39   196   786 KB
Reports/GES_DISC/CAR_lineage_evidence.json              16    21   190 KB
Reports/GES_DISC/LHSAS_lineage_evidence.json             3     5    43 KB
Reports/GES_DISC/MEaSUREs_OMI_lineage_evidence.json      4    18    73 KB
Reports/GES_DISC/MINDS_lineage_evidence.json             1     5    32 KB
Reports/GES_DISC/SNDR_lineage_evidence.json             81     0 2,127 KB
Reports/GES_DISC/TOMS_lineage_evidence

## Step 4: Verify the output

Reload each output file and check that:
- each `collections` entry, with `source_file` removed, is identical to the original file;
- there's no `suite_rollup` key, and no `source_file` is a `*_suite_lineage_evidence.json` file;
- the counts in `consolidated` are correct.

In [6]:
if DRY_RUN:
    print("Skipped: DRY_RUN=True")
else:
    failures = []
    for r in results:
        out = json.loads(r["out"].read_text())
        meta = out["consolidated"]
        if len(out["collections"]) != len(r["docs"]) or meta["collection_count"] != len(r["docs"]):
            failures.append(f"{r['out'].name}: collection count mismatch")
        for entry, (f, original) in zip(out["collections"], r["docs"]):
            entry = dict(entry)
            if entry.pop("source_file") != str(f.relative_to(REPO_ROOT)) or entry != load(f):
                failures.append(f"{r['out'].name}: entry for {f.name} differs from source")
        if "suite_rollup" in out or any(s.endswith("_suite_lineage_evidence.json") for s in meta["source_files"]):
            failures.append(f"{r['out'].name}: contains suite roll-up content")
        if meta["relationship_count"] != sum(len(c.get("relationships") or []) for c in out["collections"]):
            failures.append(f"{r['out'].name}: relationship_count mismatch")
    print(f"Checked {len(results)} output files, {sum(r['files'] for r in results)} collections: "
          f"{'all identical to their sources' if not failures else f'{len(failures)} FAILURES'}")
    for f in failures:
        print("  ", f)

Checked 51 output files, 747 collections: all identical to their sources


## Step 5: Summary

### 5a. Totals by DAAC

In [7]:
print(f"{'DAAC':<11} {'folders':>7} {'collections':>11} {'relationships':>13}")
for daac in DAACS + ["ALL"]:
    rs = [r for r in results if daac in ("ALL", r["daac"])]
    print(f"{daac:<11} {len(rs):>7} {sum(r['files'] for r in rs):>11} "
          f"{sum(r['relationships'] for r in rs):>13}")

DAAC        folders collections relationships
ASF               6          41           267
GES_DISC          9         157           320
GHRC_DAAC         1           1             0
LANCEMODIS        4          20             0
LARC_CLOUD       29         523          1037
LPCLOUD           1           3             0
LPDAAC_ECS        1           2             0
ALL              51         747          1624


### 5b. Relationships by association type and confidence

Taken from the `relationships` arrays of the per-collection files.

In [8]:
rels = [(r["daac"], rel) for r in results for _, d in r["docs"] for rel in d.get("relationships") or []]
types = Counter(rel.get("association_type") for _, rel in rels)
conf = Counter(rel.get("confidence") for _, rel in rels)

print(f"{'association_type':<20} {'count':>5}")
for t, n in types.most_common():
    print(f"{str(t):<20} {n:>5}")
print(f"\n{'confidence':<20} {'count':>5}")
for c, n in conf.most_common():
    print(f"{str(c):<20} {n:>5}")

association_type     count
RELATED                721
CHILD                  350
PARENT                 301
INPUT                  252

confidence           count
MEDIUM                1138
HIGH                   482
LOW                      4


### 5c. Sample output: `Reports/ASF/OPERA_lineage_evidence.json`

The `consolidated` block, plus the first collection entry's `source_file` and `target.short_name`.

In [9]:
sample = next(r["out"] for r in results if r["folder"] == "OPERA")
if sample.exists():
    s = json.loads(sample.read_text())
    print(json.dumps(s["consolidated"], indent=2))
    print("\ncollections[0].source_file       :", s["collections"][0]["source_file"])
    print("collections[0].target.short_name :", s["collections"][0]["target"]["short_name"])
else:
    print("(not written: DRY_RUN=True)")

{
  "generated": "2026-11-20T22:01:12+00:00",
  "notebook": "Notebooks/consolidate_lineage_evidence.ipynb",
  "data_center": "ASF",
  "group": "OPERA",
  "collection_count": 8,
  "relationship_count": 40,
  "source_files": [
    "Reports/ASF/OPERA/OPERA_L2_CSLC-S1-STATIC_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L2_CSLC-S1_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L2_RTC-S1-STATIC_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L2_RTC-S1_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L3_DISP-S1-STATIC_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L3_DISP-S1_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L3_DIST-ALERT-S1_V1_V1_lineage_evidence.json",
    "Reports/ASF/OPERA/OPERA_L4_TROPO-ZENITH_V1_V1_lineage_evidence.json"
  ]
}

collections[0].source_file       : Reports/ASF/OPERA/OPERA_L2_CSLC-S1-STATIC_V1_V1_lineage_evidence.json
collections[0].target.short_name : OPERA_L2_CSLC-S1-STATIC_V1
